In [ ]:
import sys, pathlib
project_root = pathlib.Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from trader.l2_features import build_bt_df
from trader.l3_regime import efficiency_ratio, forward_regime_label

In [ ]:
symbol = "^DJI"  # US30 - the instrument where we diagnosed the whipsaw problem
df = build_bt_df(symbol)

horizon = 20      # bars ahead to judge "was this actually a trend" (matches swing_lookback scale)
threshold = 0.219  # efficiency ratio cutoff for calling it trending

label = forward_regime_label(df["Close"], horizon=horizon, threshold=threshold)

# recent, readable slice - the full ~1700-bar history would be unreadable as one plot
window = df.iloc[-400:].copy()
window["label"] = label.reindex(window.index)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(window.index, window["Close"], color="black", linewidth=1)

ymin, ymax = window["Close"].min(), window["Close"].max()
trending_mask = window["label"] == 1.0
choppy_mask = window["label"] == 0.0

ax.fill_between(window.index, ymin, ymax, where=trending_mask, color="green", alpha=0.15, step="pre", label="labeled trending")
ax.fill_between(window.index, ymin, ymax, where=choppy_mask, color="red", alpha=0.12, step="pre", label="labeled chop")

ax.set_title(f"{symbol}: price with forward regime label (horizon={horizon}, threshold={threshold})")
ax.legend()
plt.tight_layout()
plt.show()

print("Label distribution:", label.value_counts(dropna=True).to_dict(), f" (NaN = last {horizon} bars, unlabeled)")

In [ ]:

fwd_net = (df["Close"].shift(-horizon) - df["Close"]).abs()
fwd_path = df["Close"].diff().abs().rolling(horizon).sum().shift(-horizon)
fwd_er_raw = fwd_net / fwd_path

window["fwd_er_raw"] = fwd_er_raw.reindex(window.index)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
ax1.plot(window.index, window["Close"], color="black", linewidth=1)
ax1.set_title(f"{symbol}: price")

ax2.plot(window.index, window["fwd_er_raw"], color="steelblue", linewidth=1)
ax2.axhline(threshold, color="red", linestyle="--", label=f"threshold={threshold}")
ax2.set_title(f"forward efficiency ratio, horizon={horizon}")
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
for h in [20, 40]:
    fwd_net = (df["Close"].shift(-h) - df["Close"]).abs()
    fwd_path = df["Close"].diff().abs().rolling(h).sum().shift(-h)
    fwd_er = (fwd_net / fwd_path).dropna()
    print(f"horizon={h}: median={fwd_er.median():.3f}  mean={fwd_er.mean():.3f}  "
          f"p60={fwd_er.quantile(0.6):.3f}  p70={fwd_er.quantile(0.7):.3f}  p75={fwd_er.quantile(0.75):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, h in zip(axes, [20, 40]):
    fwd_net = (df["Close"].shift(-h) - df["Close"]).abs()
    fwd_path = df["Close"].diff().abs().rolling(h).sum().shift(-h)
    fwd_er = (fwd_net / fwd_path).dropna()
    ax.hist(fwd_er, bins=40, color="steelblue")
    ax.axvline(fwd_er.median(), color="red", linestyle="--", label=f"median={fwd_er.median():.2f}")
    ax.set_title(f"horizon={h}")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from trader.l3_regime import build_regime_features

features = build_regime_features(df)
features["label"] = label  # the horizon=20, threshold=0.219 label from before

check = features.dropna().groupby("label").mean()
print(check.T)

In [ ]:
from scipy.stats import mannwhitneyu

feature_cols = ["adx_14", "er_20", "atr_expansion_50", "ema_crossover_count_20", "ma200_slope_20"]
clean = features.dropna()

for col in feature_cols:
    chop_vals = clean.loc[clean["label"] == 0.0, col]
    trend_vals = clean.loc[clean["label"] == 1.0, col]
    stat, p = mannwhitneyu(trend_vals, chop_vals, alternative="two-sided")
    auc = stat / (len(trend_vals) * len(chop_vals))
    print(f"{col:25s}  p={p:.4f}  trend-vs-chop AUC={auc:.3f}  (0.5=no separation, >0.5=trend higher, <0.5=trend lower)")